---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-11A: Coding Agents</h1>
<h2 align="center">Spec-Driven Development with Claude Code</h2>

# Learning agenda of this notebook

1. Overview of Agentic Coding Assistants
2. Claude Ecosystem and Installing Claude Code
    - Hands-On Demo 1: A Hello to Claude Code and GitHub
    - Hands-On Demo 2: A Hello to Coding Harnesses
    - Hands-On Demo 3: Accessing Harnesses via Ollama
3. Built-in Slash Commands
    - Hands-On Demo 4: Slash Commands
4. Context Window Management
    - Hands-On Demo 5: Context Window
5. The `CLAUDE.md` File
    - Hands-On Demo 6: The `CLAUDE.md` File
6. The `.claude/` Directory and the Claude Code Security Model
    - Hands-On Demo 7: Claude Code Security Model
7. Spec-Driven Development Workflow with Coding Agents

# <span style='background :lightgreen' >0. Programming Assignment: Building a Multimodal AI Assistant</span>

## [Programming Assignment 01](https://drive.google.com/file/d/1IX07R8QwcQa3IXPNGRKCGwrZalEIWMGR/view?usp=sharing)

You will build **one personal multimodal AI assistant in five versions**. Each version is a separate, self-contained public GitHub repository named `<ROLL_NO>-proj1-ver1` … `<ROLL_NO>-proj1-ver5` (for example `MSDSF26M001-proj1-ver1`). Not a branch and not a tag: you copy the previous version's code into the new repository and extend it. Every version keeps everything the earlier versions could do and adds exactly one new capability.

| Version | What the app gains | The engineering skill behind it | Weight |
|---|---|---|:---:|
| **1 · Foundation** | Multi-turn chat in English, with a local ⇄ cloud provider switch | Abstraction: one interface, two implementations | 15 |
| **2 · Memory** | Persistent conversation, user profile and editable preferences | State that survives the process | 20 |
| **3 · Voice** | Microphone input (STT) and spoken replies (TTS) | Streaming media in and out of a web app | 20 |
| **4 · Vision** | OCR and visual understanding of images and scans, with structured extraction | A specialist model against a generalist one | 25 |
| **5 · Imagination** | On-demand text-to-image generation, and the final public release | Intent detection and prompt engineering | 20 |

<div class="alert alert-block alert-info"><b>Only Version 5 is deployed.</b> Versions 1 to 4 are marked by cloning the repository and running <code>uv sync</code> and <code>uv run python app.py</code>. Version 5 is marked on its live Render URL, opened from a phone on mobile data: chat, voice, vision and image generation must all work from there. A Version 5 in which an older feature (say, the Version 3 microphone) no longer works is marked as a broken Version 5.</div>

**The architecture every version must follow:**
- **Gradio mounted inside FastAPI**, one process on one port. FastAPI adds a `GET /health` route that returns the active provider and model.
- **Three layers:** `app.py` (UI and routes, no business logic), `services/*.py` (LLM, storage, STT, TTS, vision, image generation) and `config.py` (reads the environment). **No file in `services/` may import Gradio.** This rule is graded.
- **A provider switch for every capability** (`MODEL_PROVIDER`, `STT_PROVIDER`, `TTS_PROVIDER`, `VISION_PROVIDER`, `IMAGE_PROVIDER`), each with a working local branch and a working hosted branch, and one model variable per provider (`CHAT_MODEL_OLLAMA`, `CHAT_MODEL_GROQ`, …). The local branch is not always Ollama: speech and image generation use libraries such as `transformers` or `diffusers`.
- **Render-ready from Version 1:** bind to `0.0.0.0` and the `PORT` variable, read every key and model name from the environment, commit `uv.lock` and an exported `requirements.txt`, and keep heavy local-only packages (`torch`, `transformers`, …) in a `local` dependency group, imported inside the function that uses them.

**The four ground rules:**
1. **English only.** No translation or multilingual handling.
2. **No retrieval.** No embeddings and no vector database. Memory is injected into the prompt, and a long conversation is truncated or summarised.
3. **No committed secrets.** A repository that contains an API key or a `.env` file scores **zero** for that version.
4. **Every version is separately deliverable:** five repositories, five READMEs, each runnable from a clean clone. Use `uv` throughout; never type `pip install`.

**Deliverables and marking:** six links in total, the five repository links and the live Version 5 URL. Every README has a description, a screenshot, `uv` setup steps, known limitations and a **model table** that names the model used for each task with a one-line justification. Within each version the marks are split as: new capability works 50%, earlier versions still work 15%, README 15%, both branches of every switch implemented 10%, repository hygiene 10%. Bonus: +3 for a hosted storage backend in Version 2, +3 for a hand-written HTML/CSS/JS front end, +2 for streaming replies.

**How to build it with spec-driven development and Claude Code:**
1. **Write the PRD once**, for the whole assistant: the problem, the users, the five capabilities and the success metrics.
2. **Write the TDD once**, and update it as versions are added. It must contain the architecture above, from Part A of the assignment.
3. **Write one spec per version** (`SPEC.md`, plus a `PLAN.md` with a check after every step). Each version in Part C of the assignment already gives you *What you must build* (the requirements) and *Acceptance criteria* in testable *Given … when … then …* form. Paste them into the spec, then add your edge cases.
4. **Follow the workflow for each version:** plan in plan mode → review the plan → implement → test against every acceptance criterion → commit → push. Deploy on Render for Version 5 only.
5. **Keep a `CLAUDE.md`** with the stack, the `uv` commands and the ground rules, so every new session starts with the same rules.

# <span style='background :lightgreen' >1. Overview of Agentic Coding Assistants</span>
## [Video Session 1](https://www.youtube.com/watch?v=iYRzn1-cAlw&list=PL7B2bn3G_wfBbfTiG7lCs1M5dmR2K_vVO)

## a. The Shift in Software Development

<div style="text-align:center;">
    <img src="../images/coding-agents-01.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>

## b. What is Vibe Coding?

<h3 align="center"><div class="alert alert-success" style="margin: 20px">The term <strong>vibe coding</strong> was introduced by Andrej Karpathy in February 2025. It describes a developer prompting an AI to generate software <strong>without understanding the structural requirements or the architecture</strong>, and without following any engineering process, relying on the model's assumptions to fill every gap.</div></h3>

In his <a href="https://x.com/karpathy/status/1886192184808149383" target="_blank">original post (2 February 2025)</a>, Karpathy described giving in to the vibes, accepting every change the AI suggests without reading the diff, and forgetting that the code even exists. See also <a href="https://en.wikipedia.org/wiki/Vibe_coding" target="_blank">Vibe coding on Wikipedia</a>.

<div style="text-align:center;">
    <img src="../images/coding-agents-02.png"
         style="max-width:1100px; width:100%; height:auto; display:inline-block;">
</div>


**The vibe-coding prompt.** One sentence asks for a complete product: *"Build me a full stack app for task manager with login, dashboard and analytics."* The AI answers *"Sure! Here's your application."* The prompt says nothing about:
- **Users:** who they are, and whether there are roles such as admin and member.
- **Security:** how passwords are stored and how sessions are managed.
- **Data:** which database is used and what the data model looks like.
- **Analytics:** which metrics "analytics" means, and for whom.
- **Deployment:** where the app will be deployed, and what happens when 1,000 users log in at once.

Every unanswered question becomes an assumption that the model makes silently, and two runs of the same prompt can make different assumptions.

<div style="text-align:center;">
    <img src="../images/coding-agents-03.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


**The agent loop behind it.** The developer hands a task to the AI agent. The agent figures out the steps, does the work, hits a bug, fixes it, and comes back when it is done or needs a decision. The dashed line is the only contact between the two.

This loop is very powerful, and every coding agent runs it (see Section 2). The problem with vibe coding is not the agent. It is:
- **The input:** a vague wish instead of a specification.
- **The review:** the developer only checks whether the result *seems* to work, not whether it is correct, secure and maintainable.

**When vibe coding is fine:** quick prototypes and demos, throw-away scripts, hackathons, exploring a new library, and personal tools. **When it is not:** anything that other people will use, maintain or extend.

| Limitation of vibe coding | Why it happens |
|---|---|
| **Fails in multi-team development** | There is no shared, written specification. Each developer's prompts produce different conventions, names and structures, so the parts do not fit together. |
| **Architectural inconsistency** | Each prompt is solved locally, and the model re-decides the structure every time. The codebase ends up with several patterns for the same thing. |
| **Poor scalability** | Nobody designed for growth in users, data or features, so quick fixes pile up as technical debt. |
| **Insecure and vulnerable code** | Models repeat common patterns from their training data, including insecure ones: SQL built from strings, hard-coded secrets, missing authorization checks. Nobody reviews the code before it ships. |

<div class="alert alert-warning" style="margin: 20px"><b>Vibe coding optimizes for speed of code generation, while the industry optimizes for collaboration, reliability, maintainability and scalability.</b></div>

## c. What is Spec-Driven Development (SDD)?

<div style="display:flex; justify-content:center; gap:60px; flex-wrap:wrap; text-align:center;">
    <div style="max-width:520px;">
        <img src="../images/coding-agents-04.png" style="width:100%; height:auto;">
        <p style="font-size:1.1em;"><b>In the pre-AI era, thinking was cheap and coding was expensive.</b></p>
    </div>
    <div style="max-width:520px;">
        <img src="../images/coding-agents-05.png" style="width:100%; height:auto;">
        <p style="font-size:1.1em;"><b>In the agentic era, coding is cheap and thinking is expensive.</b></p>
    </div>
</div>

- **Pre-AI era.** Implementation took weeks, so it was the expensive part. A rough idea on a whiteboard was enough to start, because the slow pace of coding left plenty of time to discover and fix gaps along the way.
- **Agentic era.** An agent implements a feature in minutes, so code is no longer the bottleneck. The scarce resource is **clear thinking**: knowing exactly what to build, for whom, under which constraints, and how to check that it works. If that thinking is not done and written down, the agent fills the gaps with guesses, which is vibe coding.

<h3 align="center"><div class="alert alert-success" style="margin: 20px"><strong>Spec-Driven Development (SDD)</strong> is an approach in which you first write down what to build and how to build it as specification documents, and the coding agent then generates code from those documents. The specs, not the chat history, are the source of truth.</div></h3>

<div style="text-align:center;">
    <img src="../images/coding-agents-06.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


| Step | Document | Answers the question | Typical contents |
|:---:|---|---|---|
| **1** | **PRD**: Product Requirements Document | *What* are we building, and *why*? | Problem statement, target users, goals, user stories, functional and non-functional requirements, scope, success metrics. No technology choices. |
| **2** | **TDD**: Technical Design Document | *How* will it be built? | Architecture, tech stack, database design, API design, security design, deployment and testing strategy. |
| **3** | **Feature Specification** | What exactly must *this one feature* do? | Description, user flows, validation rules, edge cases, boundaries, acceptance criteria. One file per feature, e.g. `.claude/specs/<feature>.md`. |
| **4** | Claude generates code | | Claude first writes a plan for the feature. You review the plan, and then Claude implements it, grounded in the PRD and TDD. |
| **5** | Review and approve | Is it done? | You check the result against the acceptance criteria, run the tests, then commit and merge. |

**Vibe coding vs spec-driven development**

| | Vibe coding | Spec-driven development |
|---|---|---|
| **Input to the agent** | A one-line wish | PRD + TDD + one feature spec |
| **Who decides the architecture** | The model, silently, on every prompt | You, once, in the TDD |
| **Memory across sessions** | The chat history, lost when the session ends | Documents stored in the repository |
| **Definition of "done"** | "It seems to work" | Every acceptance criterion passes |
| **Best for** | Solo prototypes and experiments | Teams and long-lived products |

## d. The Evolution Toward Agentic Coding

<h3 align="center"><div class="alert alert-success" style="margin: 20px"><strong>Agentic coding</strong> is an AI-driven development approach where autonomous agents can plan, reason, execute tasks, and iterate toward a defined goal while following structured workflows, engineering constraints, and system objectives.</div></h3>

| Stage | How you work with the AI | Examples |
|---|---|---|
| **Autocomplete** | The AI suggests the next line or block as you type. | GitHub Copilot (2021) |
| **Chat assistants** | You paste code into a chat window and copy the answer back into your editor. | ChatGPT (2022), Claude.ai |
| **Agentic coding** | The agent reads the repository, edits files, runs commands and tests, and iterates on its own. | Claude Code (2025), OpenAI Codex, Cursor |
| **Spec-driven agentic coding** | The agent works from written specifications inside a defined workflow, with human review gates. | Claude Code + PRD/TDD/specs, GitHub Spec Kit |

<div style="text-align:center;">
    <img src="../images/coding-agents-07.png"
         style="max-width:750px; width:100%; height:auto; display:inline-block;">
</div>


You give the agent **one request**. It **plans** the work by breaking it into tasks, **writes the code**, **tests** it by running the test suite or the app and reading the output, and **iterates, refines and delivers** until the application works. The key difference from a chat assistant is that **the agent observes the results of its own actions and corrects itself**, instead of handing every error back to you.

What agentic coding brings:
- **Automates multi-step software engineering workflows.** One request can trigger reading twenty files, editing five, running the tests and committing the change.
- **Improves productivity through task automation.** Repetitive work such as boilerplate, migrations, test writing and refactoring is delegated.
- **Integrates with tools, APIs and runtime environments.** The agent uses your shell, git, package managers and test runners, and through MCP it can use external services such as GitHub or a database.
- **Enables contextual reasoning and intelligent decision-making.** It reads the actual code, error messages and logs before deciding, instead of guessing.
- **Advances software engineering beyond code generation.** It also debugs, documents, reviews code, writes commit messages and opens pull requests.

## e. Core Components of an Agentic AI System

<div style="text-align:center;">
    <img src="../images/coding-agents-08.png"
         style="max-width:1300px; width:100%; height:auto; display:inline-block;">
</div>


| # | Component | Role | In Claude Code (covered in) |
|:---:|---|---|---|
| **1** | **LLM**: the brain | Reasons about the task, plans the steps and decides which tool to call next. | Claude Opus, Sonnet or Haiku, switched with `/model` (Sections 2 and 3) |
| **2** | **Tools**: the hands | Act on the world: read and edit files, run shell commands, search code and the web, call APIs, use git. | Built-in tools such as Read, Edit, Write, Bash, Grep, Glob, WebSearch and WebFetch (Section 2) |
| | **MCP servers**: connectors | Add tools, resources and prompts from external systems: file storage, GitHub and Jira, databases, Gmail and Slack, security tools. | `claude mcp add …` (<a href="https://modelcontextprotocol.io/" target="_blank">Model Context Protocol</a>) |
| **3** | **Context management**: the working memory | Decides what the model can see at each step: files, conversation history and project rules. | `@` mentions, `CLAUDE.md`, `/context`, `/compact` (Sections 4 and 5) |
| **4** | **Human in the loop (HITL)** | Sets the goal, reviews outputs, approves or rejects actions, and gives feedback. | Permission prompts, plan mode and permission modes (Section 6) |

**Two details worth noticing:**
- **The arrows go both ways.** The LLM calls a tool, the tool's output is added to the context, and the LLM reads it before deciding the next step. That cycle is the agent loop.
- **Codebase indexing differs between tools.** Some tools, such as Cursor, build a vector index of the repository for semantic retrieval. Claude Code does not pre-index the code with embeddings. It searches on demand with tools like Grep and Glob (*agentic search*) and reads only the files it needs.

<div class="alert alert-info" style="margin: 20px"><b>The same LLM can behave very differently depending on the tools, context and guardrails around it.</b> Everything around the model is called the <i>harness</i>. We look at harnesses in Hands-On Demo 2.</div>

## f. AI-Powered Agentic Coding Platforms

All of these platforms run the same agent loop around an LLM. They differ in **where they run** (terminal, IDE, cloud or your own server), **which models they can use** (one vendor or any provider), and **how they are licensed** (proprietary or open source).

<table style="margin-left:auto; margin-right:auto; font-size:0.95em;">
<tr><th></th><th style="text-align:left;">Platform</th><th style="text-align:left;">License</th><th style="text-align:left;">Runs in</th><th style="text-align:left;">What it is</th><th style="text-align:left;">Links</th></tr>
<tr><td style="text-align:center;"><img src="../images/coding-agents-logo-claude-code.png" width="44"></td><td style="text-align:left;"><b>Anthropic Claude Code</b><br><small>Anthropic</small></td><td style="text-align:left;">Proprietary</td><td style="text-align:left;">Terminal, IDE, desktop, web</td><td style="text-align:left;">Anthropic's agentic coder. It reads your codebase, edits files, runs commands and iterates until the task is done. Strong at deep reasoning and large refactors. <b>The tool we use in this lecture.</b></td><td style="text-align:left; white-space:nowrap;"><a href="https://claude.com/product/claude-code" target="_blank">Website</a><br><a href="https://code.claude.com/docs/en/overview" target="_blank">Docs</a></td></tr>
<tr><td style="text-align:center;"><img src="../images/coding-agents-logo-openclaw.jpeg" width="44"></td><td style="text-align:left;"><b>OpenClaw / Claw Code</b><br><small>Community</small></td><td style="text-align:left;">Open source (MIT)</td><td style="text-align:left;">Your own machine or server</td><td style="text-align:left;"><b>OpenClaw</b> is a self-hosted autonomous agent that works with any LLM and is reached through chat apps such as Telegram, Slack or WhatsApp. <b>Claw Code</b> is a community clean-room rewrite of Claude Code's architecture in Python and Rust.</td><td style="text-align:left; white-space:nowrap;"><a href="https://openclaw.ai/" target="_blank">OpenClaw</a><br><a href="https://github.com/openclaw/openclaw" target="_blank">GitHub</a><br><a href="https://claw-code.io/" target="_blank">Claw Code</a><br><a href="https://github.com/ultraworkers/claw-code" target="_blank">GitHub</a></td></tr>
<tr><td style="text-align:center;"><img src="../images/coding-agents-logo-codex.png" width="44"></td><td style="text-align:left;"><b>OpenAI Codex</b><br><small>OpenAI</small></td><td style="text-align:left;">Open-source CLI, proprietary cloud</td><td style="text-align:left;">Terminal, IDE, cloud</td><td style="text-align:left;">OpenAI's coding agent: a terminal CLI plus a cloud version that works on tasks in a sandbox and returns the results as pull requests.</td><td style="text-align:left; white-space:nowrap;"><a href="https://openai.com/codex/" target="_blank">Website</a><br><a href="https://github.com/openai/codex" target="_blank">GitHub</a></td></tr>
<tr><td style="text-align:center;"><img src="../images/coding-agents-logo-opencode.png" width="44"></td><td style="text-align:left;"><b>OpenCode</b><br><small>Community</small></td><td style="text-align:left;">Open source</td><td style="text-align:left;">Terminal, desktop, IDE</td><td style="text-align:left;">Terminal-native and model-agnostic. It works with 75+ providers, including local models through Ollama. Bring your own key, with no vendor lock-in.</td><td style="text-align:left; white-space:nowrap;"><a href="https://opencode.ai/" target="_blank">Website</a><br><a href="https://github.com/anomalyco/opencode" target="_blank">GitHub</a></td></tr>
<tr><td style="text-align:center;"><img src="../images/coding-agents-logo-pi.png" width="44"></td><td style="text-align:left;"><b>Pi</b><br><small>Mario Zechner</small></td><td style="text-align:left;">Open source (MIT)</td><td style="text-align:left;">Terminal</td><td style="text-align:left;">A minimal harness with only four tools (read, write, edit, bash) and a short system prompt. Everything else is added through extensions. It talks to 20+ providers.</td><td style="text-align:left; white-space:nowrap;"><a href="https://pi.dev/" target="_blank">Website</a><br><a href="https://github.com/earendil-works/pi" target="_blank">GitHub</a></td></tr>
<tr><td style="text-align:center;"><img src="../images/coding-agents-logo-hermes.png" width="44"></td><td style="text-align:left;"><b>Hermes Agent</b><br><small>Nous Research</small></td><td style="text-align:left;">Open source (MIT)</td><td style="text-align:left;">Your own server, CLI and chat apps</td><td style="text-align:left;">A persistent, self-hosted autonomous agent, released in February 2026, that keeps its memory and skills across sessions. A general agent that can also code.</td><td style="text-align:left; white-space:nowrap;"><a href="https://hermes-agent.nousresearch.com/" target="_blank">Website</a><br><a href="https://github.com/NousResearch/hermes-agent" target="_blank">GitHub</a></td></tr>
<tr><td style="text-align:center;"><img src="../images/coding-agents-logo-copilot.png" width="44"></td><td style="text-align:left;"><b>GitHub Copilot</b><br><small>GitHub / Microsoft</small></td><td style="text-align:left;">Proprietary</td><td style="text-align:left;">IDE, GitHub.com, cloud</td><td style="text-align:left;">A coding assistant inside VS Code, JetBrains, Visual Studio and GitHub, with agent mode, code review and a cloud coding agent that opens pull requests.</td><td style="text-align:left; white-space:nowrap;"><a href="https://github.com/features/copilot" target="_blank">Website</a></td></tr>
<tr><td style="text-align:center;"><img src="../images/coding-agents-logo-windsurf.png" width="44"></td><td style="text-align:left;"><b>Windsurf (now Devin Desktop)</b><br><small>Cognition</small></td><td style="text-align:left;">Proprietary</td><td style="text-align:left;">Desktop IDE</td><td style="text-align:left;">An AI-native IDE, formerly by Codeium. Cognition bought Windsurf in 2025 and renamed it <b>Devin Desktop</b> in June 2026. It is now a command centre for running local and cloud agents. <code>windsurf.com</code> redirects there.</td><td style="text-align:left; white-space:nowrap;"><a href="https://devin.ai/desktop" target="_blank">Website</a></td></tr>
<tr><td style="text-align:center;"><img src="../images/coding-agents-logo-cursor.png" width="44"></td><td style="text-align:left;"><b>Cursor</b><br><small>Anysphere</small></td><td style="text-align:left;">Proprietary</td><td style="text-align:left;">Desktop IDE, cloud agents</td><td style="text-align:left;">An AI-first editor forked from VS Code, with an agent inside the editor and background cloud agents for longer tasks.</td><td style="text-align:left; white-space:nowrap;"><a href="https://cursor.com/" target="_blank">Website</a></td></tr>
<tr><td style="text-align:center;"><img src="../images/coding-agents-logo-freebuff.png" width="44"></td><td style="text-align:left;"><b>Freebuff</b><br><small>Codebuff</small></td><td style="text-align:left;">Free, ad-supported</td><td style="text-align:left;">Terminal</td><td style="text-align:left;">A free terminal coding agent built on Codebuff's open multi-agent framework. It uses open-source models and is paid for by text ads shown during sessions.</td><td style="text-align:left; white-space:nowrap;"><a href="https://freebuff.com/" target="_blank">Website</a><br><a href="https://github.com/CodebuffAI/freebuff" target="_blank">GitHub</a></td></tr>
</table>

<div class="alert alert-info" style="margin: 20px"><b>Terminal agents</b> (Claude Code, Codex CLI, OpenCode, Pi, Freebuff) work in any project and any editor. <b>IDE agents</b> (Cursor, Copilot, Devin Desktop) live inside the editor. <b>Self-hosted autonomous agents</b> (OpenClaw, Hermes) run continuously on your own machine. For the rest of this lecture, and for the programming assignment, we use <b>Claude Code</b>.</div>

# <span style='background :lightgreen' >2. Claude Ecosystem and Installing Claude Code</span>
## [Video Session 2](https://www.youtube.com/watch?v=qoM3YYeny0w&list=PL7B2bn3G_wfBbfTiG7lCs1M5dmR2K_vVO)

## a. The Claude Ecosystem

<div style="text-align:center;">
    <img src="../images/coding-agents-09.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


Anthropic offers the same family of Claude models through five products, each built for a different kind of work:

| Product | What it is for | Where you use it |
|---|---|---|
| **Claude Chat** | A general assistant: ask questions, write, analyse documents and explain concepts. | claude.ai on the web, desktop app (macOS and Windows), mobile apps, Slack |
| **Claude Code** | An agentic coding assistant that works on your codebase. It reads files, edits them, runs commands and iterates. | Terminal (CLI), VS Code and JetBrains extensions, desktop app, Claude Code on the web, Slack and GitHub Actions |
| **Claude Cowork** | Collaborate with Claude on team projects: shared tasks, files and step-by-step work outside of coding. | Desktop app (macOS and Windows), with Dispatch to hand tasks over from mobile |
| **Claude Design** | Create visuals, prototypes, slide decks and one-pagers from a prompt. | claude.ai (web), with export to PPTX and PDF |
| **Claude Security** | AI security analysis: scans code for vulnerabilities and suggests fixes ranked by severity. | claude.ai/security, rolled out to Enterprise first |

## b. How to Access Each Product?

<div style="text-align:center;">
    <img src="../images/coding-agents-10.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


Each product can be reached in up to three ways: a **subscription plan**, the **API**, and **other access** channels (apps, IDEs, integrations).

- **Subscription plans:** Free (limited daily use), Pro (USD 20/month), Max 5x (USD 100/month), Max 20x (USD 200/month), Team (per seat) and Enterprise (custom pricing).
- **Claude Code is not on the Free plan.** You need Pro or higher, *or* an API key from the Anthropic Console, which is billed per token.
- **Only the models are sold through the API.** Cowork, Design and Security are packaged products that you use through their apps, not through API calls.

Prices change from time to time, so always check <a href="https://claude.com/pricing" target="_blank">claude.com/pricing</a>. For help with any product, see the <a href="https://support.claude.com/en/" target="_blank">Claude Help Center</a>.

<div class="alert alert-info" style="margin: 20px"><b>For this course:</b> a Pro subscription is enough for Claude Code. If you have no subscription, you can still practise every concept in this lecture by running Claude Code on free open models through Ollama (sub-sections i–l below).</div>

## c. How Claude Code Works

<p style="text-align:center; font-size:1.4em;"><b><i>"If you can describe it, you can build it."</i></b></p>

<div style="text-align:center;">
    <img src="../images/coding-agents-11.png"
         style="max-width:1300px; width:100%; height:auto; display:inline-block;">
</div>


1. **You provide a request.** You start Claude Code in your project folder and type a request in plain English, for example *"Add error handling to this function and write tests for it."* Claude Code can see your codebase and your terminal, and it works towards your goal.
2. **Claude Code chooses a model.**
    - **Opus** is best for complex reasoning and large code changes.
    - **Sonnet** is the best balance of intelligence and speed.
    - **Haiku** is the fastest, for simple tasks and edits.

    You can switch at any time with `/model`.
3. **Claude Code generates a response.** The response can include code changes, explanations, tests, file edits, commands that were run, and suggested next steps.
4. **Claude Code uses tools as needed.** It can read, write and edit local files, search the web, run bash commands, switch between chat, thinking and plan modes, compress past conversation, ask you questions, commit to the repository and open pull requests. It always respects your permissions, your `.gitignore` and your repository rules.

The tagline is true, but notice the word **describe**. The more precisely you describe what you want, the better the result. That is the whole idea of spec-driven development (Section 7).

## d. The Agentic Loop

<div style="text-align:center;">
    <img src="../images/coding-agents-12.png"
         style="max-width:1300px; width:100%; height:auto; display:inline-block;">
</div>


Claude Code does not just respond. It **acts, verifies and iterates until the job is done**:

1. **You enter a prompt**, for example *"Refactor this function and add tests."*
2. **Claude gathers context** by interacting with the model. The model returns either text or a **tool call** to execute.
3. **Claude takes action.** It edits files, runs commands, creates diffs and searches the code.
4. **Claude verifies the results.** It checks outputs, runs the tests and examines the changes.

**Did it achieve your goal?** If not, the loop goes round again. If yes, Claude finishes and waits for your next prompt.

**You are always in control.** The loop is:
- **Interactive:** add context, answer its questions and give feedback.
- **Interruptible:** press `Esc` to pause or stop it at any point.
- **Steerable:** type a new message while it works to guide it toward a better outcome.

| | Typical chat app | Claude Code |
|---|---|---|
| **Interaction** | Linear and one-shot: you ask, it answers | A loop that runs until the task is complete |
| **Takes action** | No. You copy the answer and run it yourself | Yes. It edits files and runs commands |
| **Checks its own work** | No | Yes. It runs tests and reads the output |
| **Stops when** | It has produced one reply | The task is complete and verifiable, or it needs your decision |

## e. Recap: The Evolution of AI Engineering

<div style="text-align:center;">
    <img src="../images/coding-agents-13.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


These are three layers of the same craft. Each one absorbs the one before it: prompting does not disappear, it becomes one component inside a larger system.

| | 1. Prompt Engineering (2022–2024) | 2. Context Engineering (2025) | 3. Harness Engineering (2026 →) |
|---|---|---|---|
| **Question** | *"How do I ask?"* | *"What does the model know at the moment it acts?"* | *"How does the agent act, check itself and stop?"* |
| **What you engineer** | Persona, task, audience, tone, output format, few-shot examples, chain-of-thought | Retrieval (RAG), short- and long-term memory, tool definitions and MCP servers, compaction, sub-agent isolation | Tool surface and sandboxed execution, persistent state, guardrails and permissions, feedback loops and evals, sub-agents and stop conditions |
| **Unit of design** | A single prompt | The context window | The whole execution environment |
| **Limit** | The model only knows what is in that one message | It curates the input, but nobody runs the loop | Agent = Model + Harness. The model is only one part |

In harness engineering you design the **agent loop** (plan → act → observe → verify) and two kinds of controls around it:
- **Guides (feed-forward)** tell the agent how to work before it acts, for example `AGENTS.md`/`CLAUDE.md`, skills and coding conventions.
- **Sensors (feedback)** tell it whether it worked, for example tests, linters, evals and code review.

<div class="alert alert-info" style="margin: 20px"><b>Where you will see this in Claude Code:</b> <code>CLAUDE.md</code> and skills are the guides (Sections 3 and 5). Tests and acceptance criteria are the sensors (Section 7). Permissions are the guardrails (Section 6).</div>

## f. Installations

**Install Claude Code** (<a href="https://claude.com/claude-code" target="_blank">claude.com/claude-code</a>, <a href="https://code.claude.com/docs/en/setup" target="_blank">setup guide</a>):

```bash
# macOS / Linux / WSL
curl -fsSL https://claude.ai/install.sh | bash
```

```powershell
# Windows PowerShell
irm https://claude.ai/install.ps1 | iex
```

On Windows, `irm` (*Invoke-RestMethod*) downloads the install script and `iex` (*Invoke-Expression*) runs it. Check where the binary was installed:

```bash
$ which claude
~/.local/bin/claude
```

If `claude` is not found:
- **macOS/Linux:** add the binary's folder to your `PATH` in `~/.zshrc` (or `~/.bashrc`), for example `export PATH="$HOME/.local/bin:$PATH"`, then run `source ~/.zshrc`.
- **Windows:** add the folder through *Edit environment variables*, then open a new terminal window.

**The other tools used in this lecture and in the assignment:**

| Tool | Why we need it | Download | Check the installation |
|---|---|---|---|
| **Claude Code** | The coding agent | <a href="https://claude.com/claude-code" target="_blank">claude.com/claude-code</a> | `claude --version` |
| **Python** | The language of our applications | <a href="https://www.python.org/downloads/" target="_blank">python.org/downloads</a> | `python --version` |
| **pip / uv** | Python package managers. The assignment uses **uv** only | <a href="https://pypi.org/project/pip/" target="_blank">pip</a> · <a href="https://pypi.org/project/uv/" target="_blank">uv</a> | `uv --version` |
| **C compiler (gcc)** | Compiles C programs. On Windows, use the MinGW-w64 build | <a href="https://sourceforge.net/projects/gcc-win64/" target="_blank">gcc-win64</a> | `gcc --version` |
| **git** | Version control, and pushing to GitHub | <a href="https://git-scm.com/install/mac" target="_blank">git for macOS</a> · <a href="https://git-scm.com/downloads" target="_blank">other OS</a> | `git --version` |
| **VS Code** | Editor, with a Claude Code extension | <a href="https://code.visualstudio.com/" target="_blank">code.visualstudio.com</a> | |
| **Node.js and npm** | JavaScript runtime. Many MCP servers and some coding agents are installed with npm | <a href="https://nodejs.org/en/download" target="_blank">nodejs.org</a> | `node --version`, `npm --version` |

<div class="alert alert-warning" style="margin: 20px"><b>Security habit:</b> <code>curl … | bash</code> and <code>irm … | iex</code> run a script from the internet with your user's rights. Only do this with official domains such as <code>claude.ai</code> or <code>ollama.com</code>, never with a link from an unknown site.</div>

## g. Three Ways to Access Claude Code

<p style="text-align:center; font-size:1.2em;"><i>Choose the login method that fits your setup</i></p>

<div style="text-align:center;">
    <img src="../images/coding-agents-14.png"
         style="max-width:1300px; width:100%; height:auto; display:inline-block;">
</div>


| | Method | Best for | How to set it up | Billing |
|:---:|---|---|---|---|
| **01** | **Claude subscription** (account login) | Students and individual developers | Run `claude`, choose *Claude account*, and sign in with your <a href="https://claude.ai" target="_blank">claude.ai</a> account (Pro or Max plan) | Included in the monthly subscription, with usage limits |
| **02** | **Anthropic Console** (API key) | Developers who want pay-as-you-go | Create a key at <a href="https://console.anthropic.com/settings/keys" target="_blank">console.anthropic.com/settings/keys</a> and save it as `ANTHROPIC_API_KEY` in an environment variable or in `~/.zshrc` | Per token |
| **03** | **Third-party cloud platform** | Companies already on AWS, Google Cloud or Azure | Amazon Bedrock, Google Vertex AI or Microsoft Foundry. See <a href="https://code.claude.com/docs/en/third-party-integrations" target="_blank">third-party integrations</a> | Through the cloud provider |

<div class="alert alert-warning" style="margin: 20px"><b>Never put an API key in your code or commit it to GitHub.</b> In the programming assignment, a repository that contains an API key scores <b>zero</b> for that version.</div>

## <span style='background :#cce5ff' >Hands-On Demo 1: A Hello to Claude Code & GitHub</span>

<div style="text-align:center;">
    <img src="../images/coding-agents-15.png"
         style="max-width:1100px; width:100%; height:auto; display:inline-block;">
</div>

## h. What is a Coding Harness?

<h3 align="center"><div class="alert alert-success" style="margin: 20px">A <strong>harness</strong> is the engineering scaffolding around an AI model (instructions, tools, environment, memory and checks) that turns an unreliable text generator into a reliable worker.</div></h3>

Every AI coding agent uses an LLM as its **brain**. The **harness** is the software wrapper around that brain: the loop, tools, permissions and workspace that turn a text-generating model into something that can take real actions. A simple rule: **if it is not the model's weights, it is the harness.**

| Part of the harness | What it provides | Example |
|---|---|---|
| **Instructions** | What the project is, its rules, and how to run it | `AGENTS.md` / `CLAUDE.md` |
| **Tools** | The ability to act: run commands, install packages, edit files, run tests | Read, Edit, Bash |
| **Environment** | A reproducible setup: locked dependencies and the correct language versions | `pyproject.toml` + `uv.lock` |
| **State** | Progress files, so the agent remembers what is done across sessions | `plan.md`, task lists |
| **Feedback** | Verification commands, so the agent checks its own work instead of guessing | `pytest`, linters |

**Why it matters:** the *same* model can produce broken output in a bare setup and excellent output in a well-built harness. When an agent fails, **fix the harness first, not the model.** A good `AGENTS.md` often beats upgrading to a more expensive model. Claude Code, Codex, OpenCode and Pi are essentially different harnesses around similar models, which is why the same model can feel very different in each of them.

**Two families of harness:**
- **Proprietary harnesses:** Claude Code and OpenAI Codex use their vendor's paid cloud models by default. They can also be pointed at free cloud or local models with `ollama launch`.
- **Open-source, model-agnostic harnesses:** OpenCode (75+ providers) and Pi (20+ providers) were built from the ground up to treat the model as a swappable, plug-in component. Both also support locally hosted models.

**tmux**, used in the next demo, is a *terminal multiplexer*. It splits one terminal window into several panes, each running an independent program, all visible on one screen. A tmux session keeps running when you detach from it, and you can reattach to it later.

| tmux command | What it does |
|---|---|
| `tmux new -s harnesses` | Start a new session named *harnesses* |
| `Ctrl-b %` / `Ctrl-b "` | Split the pane side by side / top and bottom |
| `Ctrl-b` + arrow key | Move between panes |
| `Ctrl-b d` | Detach. The session keeps running |
| `tmux attach -t harnesses` | Reattach to the session |

## <span style='background :#cce5ff' >Hands-On Demo 2: A Hello to Coding Harnesses</span>

<div style="text-align:center;">
    <img src="../images/coding-agents-16.png"
         style="max-width:1100px; width:100%; height:auto; display:inline-block;">
</div>

## i. Run Claude Code Using Ollama

<a href="https://ollama.com/" target="_blank">Ollama</a> is an open-source platform that lets you download open-source models from the Ollama library or the Hugging Face Hub, run them locally, and interact with them through a CLI, the Ollama desktop app, or its API. Since January 2026 (v0.15), it can also run agentic coding tools such as Claude Code on Ollama-served local or cloud-hosted models, through the **`ollama launch`** command (<a href="https://docs.ollama.com/integrations/claude-code" target="_blank">Ollama docs: Claude Code</a>).

<div style="text-align:center;">
    <img src="../images/coding-agents-17.png"
         style="max-width:1000px; width:100%; height:auto; display:inline-block;">
</div>


<table style="margin-left:auto; margin-right:auto; width:900px; max-width:100%;">
<tr>
<th style="text-align:center; width:50%;">Ollama cloud-hosted models</th>
<th style="text-align:center; width:50%;">Ollama local models</th>
</tr>
<tr>
<td style="text-align:center;"><img src="../images/coding-agents-logo-openai.png" width="34" style="display:inline-block; vertical-align:middle;"> &nbsp; <img src="../images/coding-agents-logo-gemini.png" width="34" style="display:inline-block; vertical-align:middle;"> &nbsp; <img src="../images/coding-agents-logo-deepseek.png" width="34" style="display:inline-block; vertical-align:middle;"></td>
<td style="text-align:center;"><img src="../images/coding-agents-logo-qwen.jpeg" width="34" style="display:inline-block; vertical-align:middle;"> &nbsp; <img src="../images/coding-agents-logo-meta-ollama.png" width="48" style="display:inline-block; vertical-align:middle;"> &nbsp; <img src="../images/coding-agents-logo-mistral.png" width="34" style="display:inline-block; vertical-align:middle;"></td>
</tr>
<tr>
<td style="text-align:left;">Run on Ollama's servers. You sign in with your Ollama account, and model names end in <code>-cloud</code> or <code>:cloud</code>. They are large and fast, and need no GPU on your laptop.</td>
<td style="text-align:left;">Downloaded to your machine and served at <code>http://localhost:11434</code>. Free, private and offline, but limited by your RAM and GPU.</td>
</tr>
</table>

**The problem this solves:** Claude Code normally needs a paid Claude subscription or API key. With `ollama launch claude`, the Claude Code harness talks to Ollama instead of Anthropic, so students without a subscription can still practise the agentic workflow.

<div class="alert alert-warning" style="margin: 20px"><b>Note:</b> this runs the coding agent on open models such as GLM or Qwen, <b>not</b> on Anthropic's actual Claude models. Expect weaker results on hard tasks. Coding agents also need a large context window: Ollama recommends setting the context length to <b>64k tokens or more</b>.</div>

## j. Install Ollama

```bash
# macOS / Linux
curl -fsSL https://ollama.com/install.sh | sh
```

```powershell
# Windows PowerShell
irm https://ollama.com/install.ps1 | iex
```

```bash
$ which ollama
/usr/local/bin/ollama
```

On macOS and Windows, the installer also includes a **GUI app** (<a href="https://ollama.com/download" target="_blank">ollama.com/download</a>). The app lets you:
- Download and pull models.
- Chat with local and cloud models.
- Drag and drop files and ask questions about them.
- Give image input, if the chosen model supports multimodality.
- Adjust the context length.

<div style="text-align:center;">
    <img src="../images/coding-agents-18.png"
         style="max-width:900px; width:100%; height:auto; display:inline-block;">
</div>


To use the **cloud models** you must sign in to Ollama (a Gmail account works). The Free plan gives limited cloud usage, and Pro (USD 20/month) and Max (USD 100/month) give more. See <a href="https://ollama.com/pricing" target="_blank">ollama.com/pricing</a> for the current limits, and browse the available models at <a href="https://ollama.com/library" target="_blank">ollama.com/library</a>.

## k. Commands to Use Ollama on the CLI

| Command | What it does |
|---|---|
| `ollama` | List the available commands |
| `ollama pull llama3.2:1b` | Download a model from the <a href="https://ollama.com/library" target="_blank">Ollama library</a> and store it locally |
| `ollama list` | List all the models downloaded to your machine |
| `ollama show llama3.2:1b` | Show the model's architecture, parameters, context window, embedding length, quantization, capabilities and licence |
| `ollama run llama3.2:1b` | Start an interactive session with a **local** model (no internet required) |
| `ollama run gpt-oss:20b-cloud` | Start an interactive session with a **cloud-hosted** model (internet required) |
| `ollama run llama3.2:1b "prompt" < file.txt` | Ask a question about a file's contents, passed through I/O redirection |
| <code>cat file.txt &#124; ollama run llama3.2:1b "prompt"</code> | The same, passing the file with the pipe operator |
| <code>curl https://www.arifbutt.me/ &#124; ollama run llama3.2:1b "question"</code> | Scrape a web page and ask a question about its HTML |
| `ollama serve` | Start the Ollama server |

After `ollama serve`, visit <b><a href="http://127.0.0.1:11434" target="_blank">http://127.0.0.1:11434</a></b>. It shows the message *"Ollama is running"*. This is the same address that the coding agents connect to.

## l. Run Harnesses via Ollama

<p style="text-align:center; font-size:1.3em;"><code>$ ollama launch &lt;agent-software&gt; --model &lt;model&gt;</code></p>

<table style="margin-left:auto; margin-right:auto; font-size:1.05em;">
<tr><th style="text-align:left;">Agent software</th><th style="text-align:left;">Launch command</th><th style="text-align:left;">Open-source models</th><th style="text-align:left;">Closed-source models</th></tr>
<tr><td style="text-align:left;"><img src="../images/coding-agents-logo-claude-code.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://claude.com/product/claude-code" target="_blank">Claude Code</a></td><td style="text-align:left;"><code>ollama launch claude</code></td><td style="text-align:left;"><img src="../images/coding-agents-logo-qwen.jpeg" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://ollama.com/library/qwen3-coder-next" target="_blank">Qwen3-coder-next</a></td><td style="text-align:left;"><img src="../images/coding-agents-logo-openai.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://developers.openai.com/api/docs/models/gpt-4.1" target="_blank">GPT-4.1</a></td></tr>
<tr><td style="text-align:left;"><img src="../images/coding-agents-logo-openclaw-red.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://openclaw.ai/" target="_blank">OpenClaw</a></td><td style="text-align:left;"><code>ollama launch openclaw</code></td><td style="text-align:left;"><img src="../images/coding-agents-logo-zai.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://ollama.com/library/glm-4.7" target="_blank">GLM-4.7:cloud</a></td><td style="text-align:left;"><img src="../images/coding-agents-logo-claude-code.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://www.anthropic.com/claude/sonnet" target="_blank">Claude Sonnet 4.6</a></td></tr>
<tr><td style="text-align:left;"><img src="../images/coding-agents-logo-codex.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://openai.com/codex/" target="_blank">Codex</a></td><td style="text-align:left;"><code>ollama launch codex</code></td><td style="text-align:left;"><img src="../images/coding-agents-logo-meta-ollama.png" width="48" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://ollama.com/library/llama3" target="_blank">Llama 3</a></td><td style="text-align:left;"><img src="../images/coding-agents-logo-gemini.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://gemini.google.com/app" target="_blank">Gemini 2.5 Pro</a></td></tr>
<tr><td style="text-align:left;"><img src="../images/coding-agents-logo-opencode.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://opencode.ai/" target="_blank">OpenCode</a></td><td style="text-align:left;"><code>ollama launch opencode</code></td><td style="text-align:left;"><img src="../images/coding-agents-logo-openai.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://ollama.com/library/gpt-oss:120b-cloud" target="_blank">gpt-oss:120b-cloud</a></td><td style="text-align:left;"><img src="../images/coding-agents-logo-xai.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://grok.com/" target="_blank">Grok 3</a></td></tr>
<tr><td style="text-align:left;"><img src="../images/coding-agents-logo-hermes.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://hermes-agent.nousresearch.com/" target="_blank">Hermes</a></td><td style="text-align:left;"><code>ollama launch hermes</code></td><td style="text-align:left;"><img src="../images/coding-agents-logo-mistral.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://mistral.ai/news/codestral-mamba" target="_blank">Codestral Mamba</a></td><td style="text-align:left;"><img src="../images/coding-agents-logo-cohere.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://cohere.com/command" target="_blank">Command R</a></td></tr>
<tr><td style="text-align:left;"><img src="../images/coding-agents-logo-pi.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://pi.dev/" target="_blank">Pi</a></td><td style="text-align:left;"><code>ollama launch pi</code></td><td style="text-align:left;"><img src="../images/coding-agents-logo-gemma.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://ollama.com/library/gemma3" target="_blank">Gemma3:27b-cloud</a></td><td style="text-align:left;"><img src="../images/coding-agents-logo-nova.png" width="30" style="display:inline-block; vertical-align:middle;"> &nbsp; <a href="https://aws.amazon.com/nova/" target="_blank">Amazon Nova Pro</a></td></tr>
</table>

<p style="text-align:center; font-size:1.2em;"><code>$ ollama launch claude | opencode | codex | pi | hermes | openclaw</code></p>

- **Agent software:** the harness you want to run. `ollama launch` configures it to talk to Ollama, then starts it. Add `--config` to configure a tool without starting it.
- **Open-source models:** served by Ollama, either locally or from Ollama's cloud (the `:cloud` tags). Any of them can be the brain of any of the harnesses on the left. For example, `ollama launch codex --model glm-4.7:cloud`.
- **Closed-source models:** the vendors' own models. They are **not** available through Ollama. A harness reaches them through the vendor's own API or subscription, for example Claude Code with your Claude account, or Codex with your OpenAI account.

See the <a href="https://docs.ollama.com/integrations/claude-code" target="_blank">Ollama integrations documentation</a> for the full list of supported tools.

## <span style='background :#cce5ff' >Hands-On Demo 3: Accessing Harnesses via Ollama</span>

<div style="text-align:center;">
    <img src="../images/coding-agents-15.png"
         style="max-width:1100px; width:100%; height:auto; display:inline-block;">
</div>

<p style="text-align:center; font-size:1.3em;"><code>$ ollama launch claude</code></p>

# <span style='background :lightgreen' >3. Built-in Slash Commands</span>
## [Video Session 3](https://www.youtube.com/watch?v=fjslw0o_pUI&list=PL7B2bn3G_wfBbfTiG7lCs1M5dmR2K_vVO)

## a. From Manual Prompts to Slash Commands

<h3 align="center"><div class="alert alert-success" style="margin: 20px"><strong>Slash commands</strong> are shortcuts you type inside a Claude Code session, starting with <code>/</code>, that trigger a specific predefined action or workflow without writing a full prompt. All coding agents support slash commands.</div></h3>

<div style="text-align:center;">
    <img src="../images/coding-agents-19.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


- **The problem: manual workflows.** You retype the same long, detailed prompt for every task: *"Explain this function: break down the inputs, outputs and logic, with clear examples…"* This is slow, repetitive, error-prone and inconsistent, because no two retyped prompts are exactly the same.
- **The shortcut.** One short word stands in for a whole prompt. Typing `/tests` expands into a full, predefined prompt (*"Write thorough unit tests with pytest, cover edge cases, and report coverage"*), and Claude runs it.
- **The solution.** A small library of commands, such as `/explain`, `/tests`, `/refactor` and `/debug`, each triggering a complete workflow.
- **The payoff.** Fast, consistent, efficient and productive. The best version of each prompt is written once and reused by everyone on the team.

## b. Four Kinds of `/` in Claude Code

<div style="text-align:center;">
    <img src="../images/coding-agents-20.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


| Kind | Who triggers it | Where it lives | Examples |
|---|---|---|---|
| **Built-in slash commands** | **You only.** The CLI runs them directly and the model is not involved | Compiled into Claude Code (60+ ship with it) | `/help`, `/model`, `/clear`, `/context`, `/compact`, `/login` |
| **Custom slash commands** | **You only**, by typing `/name` | `.claude/commands/<name>.md` (project) or `~/.claude/commands/<name>.md` (personal) | `/deploy`, `/fix-issue`, `/optimize` |
| **Built-in skills** | **You** (`/name`) **or Claude automatically**, when your request matches | Bundled with Claude Code | `/code-review`, `/debug`, `/commit`, `/pdf` |
| **Custom skills** | **You** (`/name`) **or Claude automatically** | `.claude/skills/<name>/SKILL.md` (project) or `~/.claude/skills/<name>/SKILL.md` (personal) | `/summarize-changes`, `/migrate-component` |

A custom slash command is just a Markdown file whose contents become a prompt template. `$ARGUMENTS` is replaced by whatever you type after the command name:

```markdown
<!-- .claude/commands/tests.md -->
Write thorough unit tests with pytest for $ARGUMENTS.
Cover the edge cases and report the coverage.
```

Typing `/tests services/llm.py` now runs that full prompt on `services/llm.py`.

<div class="alert alert-info" style="margin: 20px"><b>Core difference:</b> a custom slash command is <i>invoke-only</i>: nothing happens until you type <code>/name</code>. A skill adds an always-loaded description, so Claude can also trigger it on its own when the task matches, and it can bundle scripts and supporting files.</div>

<div class="alert alert-info" style="margin: 20px"><b>Why skills are now preferred:</b> skills are a superset of commands: manual <code>/name</code> plus autonomous use. They use <i>progressive disclosure</i>: only the short description sits in the context, and the body loads on demand. One unified system replaces two, and if a command and a skill have the same name, the skill wins.</div>

## c. Built-in Slash Commands

Type `/` in a Claude Code session to see the full list as you type, or see the <a href="https://code.claude.com/docs/en/commands" target="_blank">commands reference</a>.

| Group | Command | Description |
|---|---|---|
| **Help and account** | `/help` | Show usage help and the full list of available commands |
| | `/login` | Sign in to your Anthropic account |
| | `/logout` | Sign out of your Anthropic account |
| | `/status` | View the version, model, account and connectivity at a glance |
| | `/doctor` | Diagnose and verify your installation and settings |
| **Model and settings** | `/model` | Switch the AI model and save it as the default for new sessions |
| | `/effort` | Set the reasoning effort level, from low up to max |
| | `/theme` | Change the colour theme: light, dark or accessible variants |
| | `/config` | Open Settings to adjust the theme, model, output style and other preferences |
| | `/permissions` | Manage the allow, ask and deny rules for tool permissions |
| **Sessions** | `/rename` | Rename the current session and show the name on the prompt bar |
| | `/clear` | Start a new conversation with an empty context |
| | `/resume` | Reopen an earlier conversation by ID or name |
| | `/rewind` | Roll the code and/or the conversation back to an earlier checkpoint |
| | `/btw` | Ask a quick side question without adding it to the conversation |
| | `/recap` | Generate a one-line summary of the current session |
| | `/export` | Export the current conversation as plain text |
| **Context and cost** | `/context` | Visualize what is currently filling the context window |
| | `/compact` | Summarize the conversation so far to free up context |
| | `/usage` | Show the session cost, plan usage limits and activity stats |

## <span style='background :#cce5ff' >Hands-On Demo 4: Slash Commands</span>

<div style="text-align:center;">
    <img src="../images/coding-agents-15.png"
         style="max-width:1100px; width:100%; height:auto; display:inline-block;">
</div>

# <span style='background :lightgreen' >4. Context Window Management</span>

<div style="text-align:center;">
    <img src="../images/coding-agents-21.png"
         style="max-width:1000px; width:100%; height:auto; display:inline-block;">
</div>


<h3 align="center"><div class="alert alert-success" style="margin: 20px"><strong>Context</strong> is all the information the AI needs to understand your request correctly. The <strong>context window</strong> is the model's working memory: everything it can see in a single call, measured in tokens.</div></h3>

For a coding agent, context can come from your codebase, the PRD and specs, Jira issues, Slack messages, previous chats and the git history. The window is large but finite, so managing what goes into it is one of the most important skills in agentic coding.

## a. Context Window: The Mental Model

<p style="font-size:1.35em; margin-bottom:2px;"><b>It's a document passed back and forth</b></p>
<p style="font-size:1.1em; color:#555; margin-top:0;"><i>You aren't really chatting. Each turn re-sends the entire history as one input.</i></p>

<div style="text-align:center;">
    <img src="../images/coding-agents-22.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


- **Every turn, your whole history** (prompts, replies, files read, tool outputs) is bundled and sent as one big input. The model itself is stateless: it remembers nothing between calls.
- **The model reads all of it** and generates a reply, and that reply is appended to the bundle.
- **Then the whole thing is sent again** on the next turn. Nothing is dropped on its own.
- **A 10-minute irrelevant detour** still sits in the bundle and is re-sent with every single turn.

**Why the cost compounds:** message #50 re-sends messages #1–49. The longer the session, the more every new turn costs, in tokens, in money and in the model's attention.

## b. Context Window: Anatomy

<p style="font-size:1.35em; margin-bottom:2px;"><b>You see two things. Many more are loaded.</b></p>
<p style="font-size:1.1em; color:#555; margin-top:0;"><i>Behind your prompt and the reply sits a stack of hidden context, and all of it counts.</i></p>

<div style="text-align:center;">
    <img src="../images/coding-agents-23.png"
         style="max-width:1300px; width:100%; height:auto; display:inline-block;">
</div>


| Layer | What it contains | Visible to you? |
|---|---|---|
| **Hidden preamble** | Anthropic's system instructions, your `CLAUDE.md`, the built-in tool definitions, and the tool definitions of every MCP server you connected | No |
| **Your prompt** | What you typed | Yes |
| **Hidden intermediate work** | Tool calls (`Read()`, `WebSearch()`, `Write()`), the model's thinking, and the full tool results | Partly: you see summaries |
| **LLM response** | The final message | Yes |

Every MCP server you connect adds its tool definitions to the preamble of **every** call, even when you never use it. Every file Claude reads is copied into the window in full.

<div class="alert alert-info" style="margin: 20px"><b>Try it:</b> type <code>/context</code> in Claude Code to see what is filling the window right now.</div>

## c. Context Window: Growth & Degradation

<p style="font-size:1.35em; margin-bottom:2px;"><b>More context isn't more capability</b></p>
<p style="font-size:1.1em; color:#555; margin-top:0;"><i>As the window fills, the model has more to track, and quality quietly slips.</i></p>

<div style="text-align:center;">
    <img src="../images/coding-agents-24.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


**Why more context hurts.** The model spreads its attention over every token in the window. Each extra file, log or detour competes with the instructions that really matter, and facts buried in the middle of a long context are recalled less reliably than those near the start or the end. This gradual loss of quality is called **context rot**, and it begins long before the window is full.

**A practical example.** Asking Claude to `cat` a 5,000-line server log to find one error copies the whole log into the window and re-sends it on every later turn. Asking it to `grep` for the error, or handing the search to a sub-agent, brings back only the few lines that matter.

**The formula on the right is an intuition, not a law:** what matters is *correct and complete* information, not *more* information.

## d. Context Window: Auto-Compaction

<p style="font-size:1.35em; margin-bottom:2px;"><b>Auto-compaction: the automatic reset</b></p>
<p style="font-size:1.1em; color:#555; margin-top:0;"><i>Approaching the limit, Claude Code summarizes the history and continues from the summary.</i></p>

<div style="text-align:center;">
    <img src="../images/coding-agents-25.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


- **Left: full, near the limit.** Many rounds of prompts, `Read()`, `WebSearch()`, `Write()` and LLM messages are stacked on top of the preamble.
- **Middle: fresh, after compaction.** The model summarizes the conversation. The new window keeps the system instructions, `CLAUDE.md`, the tool definitions and your latest prompt, and replaces everything else with a short **memories** block.
- **Right: it works, but it is rarely ideal.**
    - A decision from three steps ago, or an early constraint, can quietly get dropped from the summary.
    - Auto-compaction usually fires in the middle of a task, when the window is already degraded, so the summary is not the model's best work.

The lesson: **do not wait for the automatic reset**. Compact or clear on purpose, at a moment you choose (next sub-section).

## e. Context Window: Take Control

<p style="font-size:1.35em; margin-bottom:2px;"><b>Compact on purpose, not by accident</b></p>
<p style="font-size:1.1em; color:#555; margin-top:0;"><i>Manual compaction runs while there's still headroom, so the summary is far better.</i></p>

<div style="text-align:center;">
    <img src="../images/coding-agents-26.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>

## f. Context Window: Persistence

<p style="font-size:1.35em; margin-bottom:2px;"><b>Let the disk hold your memory</b></p>
<p style="font-size:1.1em; color:#555; margin-top:0;"><i>Write progress to files, then start fresh sessions that read them back. Files persist; chat doesn't.</i></p>

<div style="text-align:center;">
    <img src="../images/coding-agents-27.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


1. **Session A: research.** Claude reads code and searches the web, then *"Write what we found and what's left to `research.md`."*
2. **Session B: plan.** A fresh session starts with *"Read `research.md`, then write the plan to `plan.md`."* All the research noise is gone, and only the conclusions are loaded.
3. **Session C: build.** Another fresh session: *"Read `plan.md` and continue from where we left off."*

**Why this wins:** each new session opens with a clean, full token budget, but every important decision is loaded back from disk.

<div class="alert alert-info" style="margin: 20px"><b>This is exactly what spec-driven development does.</b> The PRD, TDD, feature specs and plan files (Section 7) are the files on disk that carry the important decisions from one clean session to the next.</div>

## g. Context Window: Best Practices

<div style="text-align:center;">
    <img src="../images/coding-agents-28.png"
         style="max-width:1200px; width:100%; height:auto; display:inline-block;">
</div>


1. **One session per feature** fits spec-driven development perfectly: each feature spec gets its own fresh session, and `/resume` brings back an earlier one if you need it. Pressing `Esc` twice opens the rewind menu, so you can return to an earlier point in the conversation.
2. **Sub-agents protect the main window.** Ask, for example, *"Use a sub-agent to find every place where the API key is read."* The sub-agent reads dozens of files in **its own** context window and returns a short answer, and only that answer enters your main conversation.
3. **The 80% chart shows a trend, not an exact law.** The precise curve differs from model to model, but every model gets worse as its window fills, so leave headroom.

## h. Context Window: Cheat-Sheet

<p style="font-size:1.35em; margin-bottom:2px;"><b>Habits & command cheat-sheet</b></p>
<p style="font-size:1.1em; color:#555; margin-top:0;"><i>Three habits keep the window healthy; six commands let you steer it.</i></p>

<div style="text-align:center;">
    <img src="../images/coding-agents-29.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


Keep this page open as a reference card during the demo and while you work on the assignment. The three habits on the left matter more than any single command: **short sessions, state in files, one objective per session.**

## i. How to Give Context to Claude?

<div style="text-align:center;">
    <img src="../images/coding-agents-30.png"
         style="max-width:1300px; width:100%; height:auto; display:inline-block;">
</div>


To explore how the context window grows during a real session, see <a href="https://code.claude.com/docs/en/context-window" target="_blank">Explore the context window</a> in the Claude Code docs.

## <span style='background :#cce5ff' >Hands-On Demo 5: Context Window</span>

<div style="text-align:center;">
    <img src="../images/coding-agents-15.png"
         style="max-width:1100px; width:100%; height:auto; display:inline-block;">
</div>

# <span style='background :lightgreen' >5. The <code>CLAUDE.md</code> File</span>

## a. The Need for a `CLAUDE.md` File

<h3 align="center"><div class="alert alert-success" style="margin: 20px"><strong>CLAUDE.md</strong> is a markdown file that you normally add to the root of your project. Claude Code reads it automatically every time you start a new session.</div></h3>

<div style="text-align:center;">
    <img src="../images/coding-agents-31.png"
         style="max-width:1300px; width:100%; height:auto; display:inline-block;">
</div>


- **The problem.** Claude Code does not remember previous sessions. When you start a new session, it either reads your entire codebase again, or you have to tell it again about the architecture, project structure, tech stack, coding environment and conventions.
- **The solution.** Write that information once in `CLAUDE.md`, the single source of truth for your project. Claude Code loads it at the start of every session.
- **Two ways to create it:**
    - **Manually**, in your editor.
    - With the **`/init`** command: Claude analyzes your project and generates a `CLAUDE.md` for you to review and edit.

A short `CLAUDE.md` for the first version of the programming assignment could look like this:

```markdown
# MSDSF25M001-ver1: Multimodal AI Assistant

## Tech stack
- Python 3.11, uv, Gradio mounted inside FastAPI, OpenAI-compatible client

## Commands
- Run locally: `uv run python app.py`
- Add a package: `uv add <package>` (never `pip install`)

## Rules
- Nothing inside `services/` may import gradio
- Secrets live only in `.env`, which is never committed
```

Keep it short and factual: build commands, conventions, project layout and "always do X" rules. Everything in it is loaded into the context window of every session.

## b. Multiple `CLAUDE.md` Files

<div style="text-align:center;">
    <img src="../images/coding-agents-32.png"
         style="max-width:1300px; width:100%; height:auto; display:inline-block;">
</div>


| Location | Path | When loaded | Shared with | In git? |
|---|---|---|---|---|
| **Project root** | `./CLAUDE.md` or `./.claude/CLAUDE.md` | At the start of the session | The team | Yes |
| **Home directory** | `~/.claude/CLAUDE.md` | At the start of the session | All your projects | No |
| **Sub-directories** | e.g. `./src/CLAUDE.md` | On demand, when Claude reads files in that directory | The team | Yes |
| **Rules** | `.claude/rules/*.md` | At the start, or on demand for rules scoped to certain file paths | The team | Yes |
| **Local overrides** | `./CLAUDE.local.md` | At the start of the session | Only you | No (add it to `.gitignore`) |

Organizations can also deploy a **managed** `CLAUDE.md` that applies to every user. The folder is called **`.claude/rules/`** (the figure shows `.claude/rule/`).

These files do **not** override each other. All of them are **concatenated** into the context, from the broadest (home) to the most specific (project, then local), so the instructions closest to your work are read last. See <a href="https://code.claude.com/docs/en/memory" target="_blank">Manage Claude's memory</a>.

## c. The `/memory` Command

<div style="text-align:center;">
    <img src="../images/coding-agents-33.png"
         style="max-width:1300px; width:100%; height:auto; display:inline-block;">
</div>


`/memory` opens a menu of your memory files, so you can view and edit them without leaving Claude Code:

- **User memory:** `~/.claude/CLAUDE.md`, your personal preferences across all your projects. Not in any git repository.
- **Project memory:** `./CLAUDE.md`, project-specific instructions shared with your team through git.
- **Auto memory:** notes that Claude writes **itself**, from your corrections and preferences: snapshots, important context, session summaries, facts and decisions. They are stored per project in **`~/.claude/projects/<project>/memory/`**, with `MEMORY.md` as the index. (The figure shows `./.claude/auto-memory/`, but the real location is under your home folder, so auto memory is never committed.) `/memory` can open this folder and turn auto memory on or off.

**How Claude uses this memory:** it reads all of these sources, combines them, and uses them as context, which gives more accurate and consistent responses.

## <span style='background :#cce5ff' >Hands-On Demo 6: The <code>CLAUDE.md</code> File</span>

<div style="text-align:center;">
    <img src="../images/coding-agents-15.png"
         style="max-width:1100px; width:100%; height:auto; display:inline-block;">
</div>

# <span style='background :lightgreen' >6. The <code>.claude/</code> Directory and the Claude Code Security Model</span>

## a. Explore the `.claude/` Directory

<h3 align="center"><div class="alert alert-success" style="margin: 20px">The <strong><code>.claude/</code> directory</strong> is the central control centre and memory store for Claude Code. It stores project-specific guidelines, custom tools, permission settings and session history.</div></h3>

<div style="text-align:center;">
    <img src="../images/coding-agents-34.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


| | `.claude/` in your repository | `~/.claude/` in your home directory |
|---|---|---|
| **Scope** | This project. Committed to git and shared with the team | You. Personal, applies to every project, not tracked |
| **Files** | `CLAUDE.md`, `settings.json` (permissions, hooks, env vars, model), `settings.local.json` (your private overrides, gitignored) | `CLAUDE.md`, `settings.json` (global settings), `keybindings.json` |
| **Sub-directories** | `rules/`, `skills/`, `commands/`, `agents/` (sub-agents), `output-styles/`, `agent-memory/` | The same, plus `projects/…/memory/` (auto memory), `themes/` and `plugins/` |

Everything we discuss in Sections 3–6 (commands, skills, `CLAUDE.md`, rules, settings and permissions) lives in one of these two folders. See <a href="https://code.claude.com/docs/en/claude-directory" target="_blank">Explore the .claude directory</a>.

## b. Claude Control Center: `settings.json`

<div style="text-align:center;">
    <img src="../images/coding-agents-35.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


`settings.json` is one declarative JSON file that configures Claude Code. Most keys reload live as soon as you save the file.

| Key | What it controls |
|---|---|
| `model` | The default model for new sessions |
| `fallbackModel` | Backup models used when the primary model is overloaded |
| `outputStyle` | How Claude answers: Default, Proactive, Explanatory or Learning |
| `defaultShell` | Whether the commands you type with the `!` prefix run in `bash` or `powershell` |
| `cleanupPeriodDays` | How many days old session transcripts are kept |
| `env` | Environment variables for every session |
| `statusLine` | A custom status line under the prompt, produced by your own script |
| `permissions`, `sandbox`, `hooks` | The security model, covered next |

**Precedence (the highest source wins):** **Managed** (set by your organization; cannot be overridden) → **CLI flags** (this session only) → **Local** `.claude/settings.local.json` (yours, gitignored) → **Project** `.claude/settings.json` (team, committed) → **User** `~/.claude/settings.json` (your defaults for every project). See <a href="https://code.claude.com/docs/en/settings" target="_blank">Claude Code settings</a>.

<div class="alert alert-info" style="margin: 20px"><b>Configure it, don't hand-edit it:</b> <code>/config</code> writes your changes back to the right scope for you.</div>

## c. Permissions: Your First Line of Defense

<div style="text-align:center;">
    <img src="../images/coding-agents-36.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


Permissions are fine-grained rules that decide what Claude may do without stopping to ask. **They are enforced by Claude Code itself, not by the model**, so a confused or manipulated model cannot talk its way past them.

**Every tool call runs this gauntlet:** tool call → **deny** (hard block) → **ask** (always prompt) → **allow** (auto-approve) → **mode** (when no rule matched). The first match wins, so an allow rule cannot carve an exception out of a deny rule.

| Part of the `permissions` block | Meaning | Example |
|---|---|---|
| `defaultMode` | The fallback when no rule matches | `"default"`, `"acceptEdits"`, `"plan"`, `"auto"`, `"dontAsk"` |
| `allow` | Runs with no prompt. Syntax: `Tool(specifier)` | `"Bash(uv run *)"`, `"Bash(git status)"`, `"Edit(app/**)"` |
| `ask` | Always prompts, even if a broader allow rule also matches | `"Bash(git push *)"`, `"Read(.env*)"` |
| `deny` | Blocked in every mode | `"Bash(rm -rf *)"`, `"Bash(sudo *)"`, `"Read(secrets/**)"` |
| `additionalDirectories` | Widens file access to folders outside the project, and nothing more | `["../docs/", "../shared/"]` |

<div class="alert alert-warning" style="margin: 20px"><b>Deny beats everything.</b> A deny rule holds even in <code>bypassPermissions</code> mode. But deny rules only cover Claude's own file and Bash tools. For a real security boundary, add a <code>PreToolUse</code> hook or enable the sandbox. See <a href="https://code.claude.com/docs/en/permissions" target="_blank">Configure permissions</a>.</div>

## d. Claude Code's Permission Modes

<div style="text-align:center;">
    <img src="../images/coding-agents-37.png"
         style="max-width:1300px; width:100%; height:auto; display:inline-block;">
</div>


A **permission mode** sets the baseline for what Claude may do without asking. The rules from the previous sub-section are layered on top.

| Mode | What runs without asking | Best for |
|---|---|---|
| **Manual** (`default`) | Reads only. Claude asks before every edit or command | Watching over Claude's shoulder, sensitive work |
| **Accept Edits** (`acceptEdits`) | Reads, file edits and common filesystem commands (`mkdir`, `mv`, `cp`…) | Iterating on code you are reviewing |
| **Plan** (`plan`) | Read-only: Claude researches and proposes a plan, and changes nothing until you approve | Exploring a codebase before changing it |
| **Auto** (`auto`) | Everything, with background safety checks | Long tasks, reducing prompt fatigue |
| `dontAsk` | Only pre-approved tools. Anything that would prompt is denied | Locked-down CI pipelines and scripts |
| `bypassPermissions` ("YOLO") | Everything | Isolated containers and VMs only |

- **Switching modes:** press **`Shift+Tab`** to cycle through the modes. The active mode is shown at the bottom of your terminal. Type `/plan` to jump straight into plan mode in the middle of a conversation.
- **Recent versions:** with recent Claude Code versions, new terminal sessions **start in auto mode**, and the review-everything mode is labelled **Manual**. `bypassPermissions` is intentionally kept out of the cycle.
- **Trade-off:** more autonomy means more speed but less oversight.

See <a href="https://code.claude.com/docs/en/permission-modes" target="_blank">Choose a permission mode</a>.

## e. Using Claude Code Effectively

<div style="text-align:center;">
    <img src="../images/coding-agents-38.png"
         style="max-width:1200px; width:100%; height:auto; display:inline-block;">
</div>


1. **The context window.** Think of it as Claude's working memory. It can hold a lot, but not everything at once. This is where the *agentic* part comes in: Claude finds strategic ways to locate answers in your codebase without loading the whole thing into context.
2. **It asks for permission.** By default, Claude Code asks you before running commands or making changes. You choose how hands-on you want to be, from reviewing and approving each step to letting Claude proceed with safeguards. You are always in control.
3. **It can make mistakes.** Like any tool, Claude Code is not perfect. It can misunderstand your intent, introduce a bug or over-engineer a solution (note the `=` instead of `==` in the figure). **Stay in the loop:** review the changes, test your code, ask questions and catch issues early.

**The sweet spot:** use Claude Code's power, keep oversight, and collaborate to build better software, faster and safer.

## <span style='background :#cce5ff' >Hands-On Demo 7: Claude Code Security Model</span>

<div style="text-align:center;">
    <img src="../images/coding-agents-15.png"
         style="max-width:1100px; width:100%; height:auto; display:inline-block;">
</div>

# <span style='background :lightgreen' >7. Spec-Driven Development Workflow with Coding Agents</span>

## [Video Session 4](https://www.youtube.com/watch?v=9dTnjDKtxS4&list=PL7B2bn3G_wfBbfTiG7lCs1M5dmR2K_vVO)
## [Spec Driven Development Simulator](https://arifpucit.github.io/cc-workflow/)

## a. What is Spec-Driven Development (SDD)?

<div style="text-align:center;">
    <img src="../images/coding-agents-39.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


The figure walks through a real example: **adding authentication to a web app**, built as 1 PRD → 1 TDD → N feature specs.

1. **PRD (1 document):** the business requirement, the *why*. *"A user must be authenticated before accessing the service."* It describes the user-facing outcome, with no technical details.
2. **TDD (1 document):** the *how*. Signup with email and bcrypt hashing, login with JWT and refresh tokens, session storage, password reset, security controls (rate limiting, CSRF, OWASP), the data model and the API contracts.
3. **Feature specs (N documents):** the TDD is cut into self-contained, implementable units. For authentication, that gives five specs: **Signup, Login, Password Reset, Session Management and Logout**. Each has its form and validation rules, success path, edge cases and acceptance criteria.
4. **Claude generates the code** for one feature spec at a time, grounded in the PRD and TDD, and you **review and approve** it against that spec's acceptance criteria.

<div class="alert alert-info" style="margin: 20px"><b>Key insight:</b> the number of feature specs is <b>not fixed</b>. It depends on how the TDD decomposes the system: five for authentication, perhaps two or three for a small MVP, ten or more for an enterprise system. <b>Rule of thumb: one feature spec = one self-contained, shippable unit of work that Claude can implement in a single focused session.</b></div>

## b. Spec-Driven Development with Claude Code

<div style="text-align:center;">
    <img src="../images/coding-agents-40.png"
         style="max-width:1300px; width:100%; height:auto; display:inline-block;">
</div>


<p style="text-align:center; font-size:1.3em;"><b>Plan it. Build it. Test it. Ship it. Repeat!</b></p>

| Step | What happens | Who |
|:---:|---|---|
| 1–3 | Prepare the **PRD**, the **TDD** and the **spec file** for one feature | You, helped by Claude |
| 4–5 | Give the documents to Claude, and Claude **generates a plan file** | Claude, in plan mode |
| 6 | **Review the plan.** Fix it now: it is much cheaper than fixing code later | You |
| 7–8 | Execute the plan. Claude **generates the code** | Claude |
| 9 | **Test it** against the spec's acceptance criteria | You and Claude |
| 10–11 | **Commit** the changes and **push** to GitHub | You (or Claude, with your approval) |

GitHub's open-source <a href="https://github.com/github/spec-kit" target="_blank">Spec Kit</a> packages this same workflow as ready-made slash commands for Claude Code and other agents: `/speckit-constitution` (project principles), `/speckit-specify` (the spec), `/speckit-plan` (the technical plan), `/speckit-tasks` (a task list) and `/speckit-implement`.

## c. The Feature Implementation Workflow

<div style="text-align:center;">
    <img src="../images/coding-agents-41.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


The workflow has three documents and one repeated loop. See it interactively at <a href="https://arifpucit.github.io/cc-workflow/" target="_blank">arifpucit.github.io/cc-workflow</a>.

- **Step 1, the PRD,** defines **what** needs to be built: problem statement, business goals, user requirements, functional and non-functional requirements, constraints, epics (user stories), acceptance criteria, success metrics, scope and risks.
- **Step 2, the TDD,** defines **how** it will be built: architecture, technology stack, database design, API design, security design, deployment strategy and testing strategy. Evaluate the alternatives, then finalize the design.
- **Step 3, the feature specification** (`.claude/specs/<feature-name>.md`), defines **one** feature in detail: description, requirements, boundaries and acceptance criteria.

**The detailed workflow (steps 1–13) is repeated for every feature.** The parts that matter most:

| Steps | Why they matter |
|---|---|
| **Create a branch** (2) | Each feature is isolated. If Claude's work goes wrong, `main` is untouched |
| **Plan with Claude Code** (3) | Claude writes a plan from the spec *before* any code. Reviewing a plan is cheaper than reviewing code |
| **Test the feature** (5) | The spec's acceptance criteria are the test list. "It runs" is not the same as "it passes" |
| **Pull request, code review, address feedback** (8–10) | A second pair of eyes, human or AI, before the code reaches `main` |
| **CI/CD, merge, deploy** (11–13) | Automated checks run on every change, and only reviewed code is deployed |

Then pick the next feature from the PRD or backlog and repeat. When all the features are complete, a high-quality product has been delivered.

**Document flow:** PRD (what to build) → TDD (how to build it) → Feature spec (what this feature must do) → Plan (how Claude will implement it) → Code (the actual implementation) → Test (verify it works).

## d. PRD (Product Requirements Document)

<div style="text-align:center;">
    <img src="../images/coding-agents-42.png"
         style="max-width:1100px; width:100%; height:auto; display:inline-block;">
</div>


<h3 align="center"><div class="alert alert-success" style="margin: 20px"><strong>Purpose:</strong> defines what the product is, why it exists and who it serves.</div></h3>

Our example is an **Educational Security Lab**: a deliberately vulnerable web app (signup and login, a protected dashboard, user search, logout) with **8 intentional vulnerabilities** such as SQL injection and XSS, so that students learn by doing: *exploit → understand → fix*. Its PRD covers:

| Section | Answers |
|---|---|
| Problem Statement | What problem are we solving? |
| Proposed Solution | What will we build? |
| Target Audience | Who is it for? (students, trainees, instructors) |
| Core Features | What can a user do? |
| 8 Intentional Vulnerabilities | Which weaknesses are built in, on purpose? |
| Non-Functional Requirements | How fast, how reliable, where it runs |
| Learning Outcomes | What will a student be able to do afterwards? |
| Success Metrics | How do we know it worked? |

Read the complete PRD: <a href="https://github.com/arifpucit/vuln-web-app/blob/main/docs/PRD.md" target="_blank">vuln-web-app/docs/PRD.md</a>

## e. TDD (Technical Design Document)

<div style="text-align:center;">
    <img src="../images/coding-agents-43.png"
         style="max-width:1100px; width:100%; height:auto; display:inline-block;">
</div>


<h3 align="center"><div class="alert alert-success" style="margin: 20px"><strong>Purpose:</strong> explains the architecture, technologies, components, data flow and technical implementation. <i>The PRD says WHAT. The TDD says HOW.</i></div></h3>

| Section | Answers |
|---|---|
| System Architecture | The big-picture structure |
| Technology Stack | Which languages, frameworks and tools |
| Backend Components | The server-side building blocks |
| Frontend Components | The pages and UI pieces |
| Database Design | The tables, fields and relationships |
| Application Data Flow | How a request moves through the system |
| Security Vulnerability Design | Exactly how each intentional vulnerability is implemented |
| Technical Requirements | Versions, environment and constraints |
| Testing & Verification | How each part is checked |

Read the complete TDD: <a href="https://github.com/arifpucit/vuln-web-app/blob/main/docs/TDD.md" target="_blank">vuln-web-app/docs/TDD.md</a>

## f. Spec File

<div style="text-align:center;">
    <img src="../images/coding-agents-44.png"
         style="max-width:1100px; width:100%; height:auto; display:inline-block;">
</div>


<h3 align="center"><div class="alert alert-success" style="margin: 20px"><strong>Purpose:</strong> defines the exact system behaviour, user flows, validation rules and UI requirements for one feature. <i>Clear specs. Fewer bugs. Happy everyone!</i></div></h3>

| Section | Answers |
|---|---|
| Application Runtime Behavior | What the app does when it starts and runs |
| User Flows | The step-by-step journeys, from signup to dashboard |
| Functional Requirements | What the system must do |
| UI Specifications | What each screen shows |
| Session State & Data Lifecycle | What is stored, where and for how long |
| Success Paths & Alternate Paths | The happy path, and the other valid paths |
| Edge Cases | What happens when things go unexpectedly |
| Acceptance Criteria & Testing | The checks that prove the feature is done |

## g. Documents to Production: Workflow

The 11 steps in the figure in sub-section b take a feature from documents to production. In Claude Code, one feature looks like this:

| # | Step | In practice |
|:---:|---|---|
| 1 | Start clean, on a branch | `git checkout -b feature/<name>`, then start `claude` (or `/clear` in a running session) |
| 2 | Plan | Press `Shift+Tab` to enter plan mode: *"Read `docs/PRD.md`, `docs/TDD.md` and `.claude/specs/<name>.md`. Write an implementation plan to `.claude/specs/<name>-plan.md`."* |
| 3 | Review the plan | Read it and correct it. Ask Claude to revise it until it matches the spec |
| 4 | Build | *"Implement `.claude/specs/<name>-plan.md`."* |
| 5 | Test | Run the app and the tests. Check every acceptance criterion in the spec, and ask Claude to fix any that fail |
| 6 | Commit and push | `git add`, `git commit`, `git push`, then open a pull request |
| 7 | Next feature | `/clear`, and repeat from step 1 with the next spec |

## h. Programming Languages & Tech Stacks

<div style="text-align:center;">
    <img src="../images/coding-agents-45.png"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>


A coding agent can write in any of these languages, but **you** choose the stack in the TDD. Here is how the layers connect:

- **Frontend** runs in the browser: **HTML** for structure, **CSS** for style and **JavaScript** for interactivity, often with frameworks such as React, Vue or Tailwind.
- **Backend** runs on the server: business logic, APIs, authentication and data access, in **Python** (Django, Flask, FastAPI), **Java** (Spring Boot), **PHP** (Laravel), **Go** or **.NET (C#)**.
- **Systems and mobile** run close to the hardware: **C**, **C++** and **Rust** for operating systems and performance-critical software, and **Objective-C** and **Swift** for Apple platforms.
- **Databases** are chosen by the shape of the data:
    - **SQL** for structured data and transactions (PostgreSQL, MySQL, SQLite).
    - **NoSQL** for flexible schemas (MongoDB, Redis, Neo4j).
    - **Vector** databases for embeddings and RAG (Chroma, Pinecone, pgvector).

**For the programming assignment:** a Python backend with **FastAPI**, a **Gradio** user interface mounted inside it, JSON files for storage, and a vector store in Version 6.

## i. Deployment Platforms

<div style="text-align:center;">
    <img src="../images/coding-agents-46.png"
         style="max-width:800px; width:100%; height:auto; display:inline-block;">
</div>


| Platform | Best for |
|---|---|
| <a href="https://www.hostinger.com/pk/pricing/vps-hosting" target="_blank">Hostinger</a> | Shared and VPS hosting: you get a server and manage it yourself |
| <a href="https://render.com/" target="_blank">Render</a> | Deploys web services straight from a GitHub repository, with a free tier. <b>Used in the assignment</b> |
| <a href="https://huggingface.co/spaces" target="_blank">Hugging Face Spaces</a> | Hosts ML demos built with Gradio, Streamlit or Docker |
| <a href="https://aws.amazon.com/free/webapps/" target="_blank">AWS Cloud</a> | The largest cloud: from simple web apps to fully custom infrastructure |
| <a href="https://vercel.com/" target="_blank">Vercel</a> | Front-end and serverless hosting, especially for Next.js and React apps |
| <a href="https://cloud.google.com/" target="_blank">Google Cloud</a> | Google's cloud: Cloud Run, App Engine, Compute Engine and Vertex AI |

## j. Deployment: GitHub → Render

<div style="text-align:center;">
    <img src="../images/coding-agents-47.png"
         style="max-width:1300px; width:100%; height:auto; display:inline-block;">
</div>


**The deployment flow:** the developer writes code and runs `git push` → the commit lands on the `main` branch on GitHub → a webhook tells **Render** to auto-deploy → Render installs the dependencies and starts Uvicorn → the app is live over HTTPS at `https://<name>.onrender.com`.

**One-time setup, in six steps:**
1. Push your project, including `requirements.txt`, to GitHub.
2. In the Render dashboard, click **New + → Web Service**.
3. Connect your GitHub account, then choose the repository and the branch (usually `main`).
4. Set the name, region and runtime (Python 3), the **build command** and the **start command**.
5. Under the advanced settings, add your **environment variables** (API keys) and, on paid instances, a persistent disk.
6. Click **Create Web Service** and watch the events log until the status turns green and shows **Live**.

**For the programming assignment** (Gradio mounted inside FastAPI, managed with uv), use the settings the assignment gives:

| Render setting | Value |
|---|---|
| Runtime | Python 3 |
| Build command | `pip install uv && uv pip install --system -r requirements.txt` (plain `pip install -r requirements.txt` is an acceptable fallback) |
| Start command | `python app.py`, which calls `uvicorn.run(app, host="0.0.0.0", port=int(os.environ.get("PORT", 7860)))` |
| Environment variables | `MODEL_PROVIDER`, your API keys and model names. Never in the repository |

`requirements.txt` is exported from `uv.lock` with `uv export --format requirements-txt --no-hashes --no-group local --no-emit-project -o requirements.txt`. If you hard-code the port or bind to `127.0.0.1`, Render reports *"no open ports detected"* and the deploy fails.

<div class="alert alert-warning" style="margin: 20px"><b>Free-tier limits (these are not bugs):</b> the service sleeps after 15 minutes of inactivity, so the first request afterwards takes about a minute. It has 512 MB of RAM, a filesystem that is wiped on every redeploy, and no system packages. Never load a model into the web service; call a hosted API instead.</div>

<br>
<div style="text-align:center;">
    <img src="../images/coding-agents-48.png"
         style="max-width:450px; width:100%; height:auto; display:inline-block;">
</div>

<br>